# TGIF-Train-Clean-Subset Cohort Acquisition Launcher (Colab CPU)

> **Phase**: Phase 4C.7B trace — Independent Cohort Acquisition (Existing Benchmark Intake)
> **Execution Environment**: Google Colab Standard CPU Runtime (0 GPU required, 0 VRAM)
> **Approved Scope**: INTAKE ONLY (Download 2 archives ~18.63 GiB, selective extract 400 pairs, package ~120-160 MB ZIP return to local. Detector, evaluation, and training NOT AUTHORIZED)
> **Approved Cohort**: $N=400$ pairs (14 Large [3.5%], 221 Medium [55.25%], 165 Small [41.25%])
> **Locked Manifest SHA-256**: `53a6ee472fe840a42abd97ccb7475932e0720f5788f745f57f7a2bcfbc32cc8c`

---

In [ ]:
# Cell 1: Environment & Disk Capacity Preflight
import os
import shutil
import sys

print("=== Cell 1: Environment Preflight ===")
print(f"Python Version: {sys.version}")
assert sys.version_info >= (3, 10), "Python 3.10+ required."

total, used, free = shutil.disk_usage("/content")
free_gib = free / (1024 ** 3)
print(f"Colab /content Disk: Total = {total / (1024**3):.1f} GiB, Free = {free_gib:.1f} GiB")
if free_gib < 25.0:
    raise RuntimeError(f"Insufficient disk space ({free_gib:.1f} GiB < 25.0 GiB) to download 18.63 GiB archives.")

!pip install -q Pillow numpy
import numpy as np
import PIL
print(f"Pillow Version: {PIL.__version__}")
print(f"NumPy Version: {np.__version__}")
print("Environment preflight PASS.")

In [ ]:
# Cell 2: Repository Clone & Locked Manifest Verification
import hashlib
from pathlib import Path

print("=== Cell 2: Repository Clone & Manifest Verification ===")
REPO_DIR = Path("/content/forensics-web-lab")
BRANCH = "research/independent-cohort-acquisition"
EXPECTED_LOCKED_MANIFEST_SHA256 = "53a6ee472fe840a42abd97ccb7475932e0720f5788f745f57f7a2bcfbc32cc8c"

if not REPO_DIR.exists():
    !git clone --branch {BRANCH} https://github.com/nomozer/forensics-web-lab.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}

manifest_path = REPO_DIR / "research/evidence/phase-4c.7b/tgif_train_clean_subset_manifest_locked_n400.json"
assert manifest_path.is_file(), f"Manifest not found: {manifest_path}"

with open(manifest_path, "rb") as f:
    actual_sha = hashlib.sha256(f.read()).hexdigest()

print(f"Manifest Path: {manifest_path}")
print(f"Actual SHA-256: {actual_sha}")
assert actual_sha == EXPECTED_LOCKED_MANIFEST_SHA256, (
    f"Manifest SHA mismatch: {actual_sha} != {EXPECTED_LOCKED_MANIFEST_SHA256}"
)
print("Locked Manifest Verification PASS: Exact hash match.")

In [ ]:
# Cell 3: Download TGIF Training Archives from Official Nextcloud
import time
import urllib.request

print("=== Cell 3: Download TGIF Archives (18.63 GiB) ===")
ARCHIVES = {
    "orig_training.tar.gz": {
        "url": "https://cloud.ilabt.imec.be/index.php/s/xEeAzrY7ES9KA8o/download?path=%2Forig&files=orig_training.tar.gz",
        "expected_bytes": 5652563073,  # ~5.26 GiB
    },
    "sd2-sp_training.tar.gz": {
        "url": "https://cloud.ilabt.imec.be/index.php/s/xEeAzrY7ES9KA8o/download?path=%2Fsd2-sp&files=sd2-sp_training.tar.gz",
        "expected_bytes": 14359902282,  # ~13.37 GiB
    },
}

content_dir = Path("/content")
for filename, info in ARCHIVES.items():
    dest = content_dir / filename
    expected = info["expected_bytes"]
    if dest.is_file() and dest.stat().st_size == expected:
        print(f"Archive already present and size verified: {filename} ({expected} bytes)")
        continue

    print(f"Downloading {filename} ({expected / (1024**3):.2f} GiB)...")
    part = dest.with_suffix(dest.suffix + ".part")
    t0 = time.time()
    req = urllib.request.Request(info["url"], headers={"User-Agent": "ForensicsWebLab-ColabIntake/1.0"})
    with urllib.request.urlopen(req) as resp, open(part, "wb") as out_f:
        dl = 0
        last_p = t0
        while chunk := resp.read(8 * 1024 * 1024):
            out_f.write(chunk)
            dl += len(chunk)
            now = time.time()
            if now - last_p > 15.0:
                rate = (dl / (1024 * 1024)) / max(now - t0, 0.001)
                print(f"  {filename}: {dl / (1024**3):.2f} / {expected / (1024**3):.2f} GiB ({rate:.1f} MB/s)")
                last_p = now
    part.rename(dest)
    assert dest.stat().st_size == expected, f"Size mismatch for {filename}: {dest.stat().st_size} != {expected}"
    print(f"Completed {filename} in {time.time() - t0:.1f}s.")

print("All archives downloaded and size verified.")

In [ ]:
# Cell 4: Selective Stream Extraction & Package Creation
import subprocess

print("=== Cell 4: Execute Selective Extraction Worker ===")
cmd = [
    sys.executable,
    str(REPO_DIR / "scripts/research/acquire_tgif_train_subset_colab.py"),
    "--manifest", str(manifest_path),
    "--archive-dir", "/content",
    "--output-zip", "/content/tgif_train_clean_subset_package.zip",
]

res = subprocess.run(cmd, capture_output=False, text=True)
if res.returncode != 0:
    raise RuntimeError(f"Intake worker failed with exit code {res.returncode}.")
print("Worker execution PASS: Package zip generated.")

In [ ]:
# Cell 5: Package Verification & Handover Instructions
import json
from google.colab import files

print("=== Cell 5: Package Handover ===")
package_zip = Path("/content/tgif_train_clean_subset_package.zip")
assert package_zip.is_file(), f"Package not found: {package_zip}"

zip_bytes = package_zip.stat().st_size
with open(package_zip, "rb") as f:
    zip_sha = hashlib.sha256(f.read()).hexdigest()

print(f"Package File: {package_zip.name}")
print(f"Size: {zip_bytes / (1024 * 1024):.2f} MB ({zip_bytes:,} bytes)")
print(f"SHA-256: {zip_sha}")

print("\n--- NEXT STEPS ON LOCAL WORKSTATION ---")
print("1. Download 'tgif_train_clean_subset_package.zip' to local workstation")
print("   (into folder: data/research/local-artifacts/phase-4c.7b/)")
print("2. Run local forensic audit and tripartite verification:")
print("   python scripts/research/audit_tgif_train_subset_local.py --package-zip data/research/local-artifacts/phase-4c.7b/tgif_train_clean_subset_package.zip")
print("3. Stop before evaluation to inspect the generated contact sheet.")

# Trigger browser download in Colab
files.download(str(package_zip))